# v3.4 MaskDINO/Co-DINO Sidecar Attempt

MaskDINO requires `detectron2`; Co-DINO requires OpenMMLab `mmcv`.
Both are incompatible with torch>=2.2 + CUDA 13.0 in the current env.

**Result:** `sidecar_required` — expert conda isolation needed

In [ ]:
# Document sidecar blockers
blockers = {
    'maskdino-r50-coco': 'detectron2 build chain incompatible with torch>=2.2 + cu130',
    'co-dino-inst-vit-l-coco': 'OpenMMLab mmcv2.1 incompatible with current torch version',
    'oneformer-dinat-large': 'NATTEN has no prebuilt wheel for torch>=2.2',
    'rtdetrv4-s': 'Google Drive gdown abuse filter blocks checkpoint download',
}
for mid, reason in blockers.items():
    print(f'{mid}:\n  blocker: {reason}\n')

## Sidecar Isolation Commands

### MaskDINO
```bash
conda create -n maskdino python=3.10
conda activate maskdino
pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.1/index.html
git clone https://github.com/IDEACVR/MaskDINO
pip install -e MaskDINO
```

### Co-DINO
```bash
conda create -n codino python=3.10
conda activate codino
pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
pip install mmdet
git clone https://github.com/Sense-X/Co-DETR && pip install -e Co-DETR
```

In [ ]:
# Confirm sidecar models are not accidentally benchmark_passed
from visionservex.model_zoo import get_model_source, SOURCE_MANIFEST
for mid in ['maskdino-r50-coco', 'co-dino-inst-vit-l-coco']:
    src = get_model_source(mid)
    if src:
        print(f'{mid}: recommended_action={src.recommended_action}')
        assert src.recommended_action not in ('add_now',)
    else:
        print(f'{mid}: not in SOURCE_MANIFEST (expected — sidecar only)')

In [ ]:
# HQ-SAM legal review status
from visionservex.vsx import VSX, VSXError
for mid in ['hq-sam', 'edge-sam']:
    info = VSX.sam(mid).explain()
    print(f'{mid}: state={info["state"]} (expected: legal_review_required or excluded_restricted)')